## LG Aimers 9기 — 두 번째 전체 학습 및 제출 ZIP (Colab)

05b에서 2022·2023·2024 모두 개선된 **LightGBM pitchmix_shape + CatBoost all_new**를 전체 2019~2024 데이터로 학습한다. 첫 제출과 같은 65:35 logit 평균을 사용하고 별도 보정은 적용하지 않는다. 최종 출력은 DACON 코드 제출용 `submit_02.zip`이다.

### 0. Drive 마운트와 프로젝트 확인

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("Colab이 아니므로 Drive 마운트를 건너뜁니다.")

candidates = [
    Path("/content/drive/MyDrive/LG_aimers_9th"),
    Path("/content/drive/MyDrive/LG Aimers 9th"),
    Path.cwd(),
]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "experiments/05c_train_submit.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "LG_aimers_9th 폴더 또는 experiments/05c_train_submit.py를 찾지 못했습니다."
    )

required = [
    PROJECT_ROOT / "data/train.csv",
    PROJECT_ROOT / "data/test.csv",
    PROJECT_ROOT / "data/sample_submission.csv",
    PROJECT_ROOT / "src/features.py",
    PROJECT_ROOT / "src/submission_entrypoint.py",
    PROJECT_ROOT / "experiments/04_train_submit.py",
    PROJECT_ROOT / "experiments/05c_train_submit.py",
    PROJECT_ROOT / "outputs/oof_recombination_v1/latest_run.json",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"필수 파일이 없습니다: {missing}")

import numpy as np
import pandas as pd

print("PROJECT_ROOT:", PROJECT_ROOT)
print("train.csv size (GB):", round((PROJECT_ROOT / "data/train.csv").stat().st_size / 1e9, 3))

### 1. 모델 패키지와 GPU 확인

Colab 기본 NumPy·pandas는 다시 설치하지 않고 LightGBM과 CatBoost만 설치한다. A100은 CatBoost 전체 학습에 사용하며 LightGBM은 CPU로 학습한다.

In [ ]:
install = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--disable-pip-version-check", "-q",
        "lightgbm==4.6.0", "catboost==1.2.8",
    ],
    capture_output=True, text=True,
)
if install.returncode != 0:
    print(install.stdout)
    print(install.stderr)
    raise RuntimeError("모델 패키지 설치에 실패했습니다.")

import lightgbm
import catboost

gpu_binary = shutil.which("nvidia-smi")
gpu_check = (
    subprocess.run([gpu_binary, "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
    if gpu_binary else None
)
gpu_available = gpu_check is not None and gpu_check.returncode == 0
print("LightGBM:", lightgbm.__version__)
print("CatBoost:", catboost.__version__)
print("GPU:", gpu_check.stdout.strip() if gpu_available else "not detected")
if not gpu_available:
    print("경고: CPU 실행도 가능하지만 CatBoost 학습이 느려집니다.")

### 2. 05b 선택 결과 재확인

학습 전에 05b가 선택한 조합이 정확히 `lgb_pitchmix_shape__cat_all_new`인지, 2023·2024가 모두 base보다 좋아졌는지 다시 확인한다. 혼합비 grid의 낙관적 1등은 사용하지 않는다.

In [ ]:
OOF_LATEST_PATH = PROJECT_ROOT / "outputs/oof_recombination_v1/latest_run.json"
with OOF_LATEST_PATH.open(encoding="utf-8") as handle:
    oof_latest = json.load(handle)
oof_summary = oof_latest["summary"]
OOF_RUN_DIR = Path(oof_latest["run_dir"])
fixed_summary = pd.read_csv(OOF_RUN_DIR / "fixed_pair_summary.csv")
selected = fixed_summary[
    fixed_summary["pair"].eq("lgb_pitchmix_shape__cat_all_new")
]
if len(selected) != 1:
    raise RuntimeError("05b 선택 조합 행을 찾지 못했습니다.")
selected = selected.iloc[0]
if oof_summary["selected_safe_fixed_pair"] != "lgb_pitchmix_shape__cat_all_new":
    raise RuntimeError("05b 자동 선택 결과가 예상과 다릅니다.")
if not bool(selected["safe_2023_2024"]):
    raise RuntimeError("선택 조합이 2023·2024 안전 조건을 통과하지 못했습니다.")
display(selected[[
    "pair", "recent_brier", "recent_bss_raw",
    "brier_2022", "brier_2023", "brier_2024",
    "delta_recent_brier_vs_base", "delta_brier_2023_vs_base",
    "delta_brier_2024_vs_base", "safe_2023_2024",
]])

### 3. 최종 학습 설정

LightGBM은 R 경기·half-life 2.5·500회와 pitchmix 피처 10개를 사용한다. CatBoost는 R·F 전체·half-life 0.5·85회와 05의 새 피처 47개를 모두 사용한다. CatBoost의 `game_type`은 아직 넣지 않으며, 별도 확률 보정도 하지 않는다.

In [ ]:
THREADS = max(1, min(8, os.cpu_count() or 2))
CATBOOST_TASK_TYPE = "GPU" if gpu_available else "CPU"
LGBM_ESTIMATORS = 500
CATBOOST_ITERATIONS = 85
OUTPUT_DIR = PROJECT_ROOT / "outputs/final_submission_v2"

settings = {
    "selected_pair": "lgb_pitchmix_shape__cat_all_new",
    "lgbm": "R-only / half-life 2.5 / pitchmix_shape / 500 trees",
    "catboost": "R+F / half-life 0.5 / all_new / 85 iterations",
    "blend": "65:35 logit / no calibration",
    "catboost_task_type": CATBOOST_TASK_TYPE,
    "output": str(OUTPUT_DIR),
}
settings

### 4. 입력 데이터와 제출 행 순서 사전 점검

In [ ]:
train_check = pd.read_csv(
    PROJECT_ROOT / "data/train.csv",
    usecols=["row_id", "season", "game_type", "control_success"],
    encoding="utf-8-sig",
)
test_ids = pd.read_csv(
    PROJECT_ROOT / "data/test.csv", usecols=["row_id"], encoding="utf-8-sig"
)
sample_ids = pd.read_csv(
    PROJECT_ROOT / "data/sample_submission.csv",
    usecols=["row_id"], encoding="utf-8-sig",
)
if train_check["row_id"].isna().any() or train_check["row_id"].duplicated().any():
    raise ValueError("train row_id가 유일하지 않습니다.")
if len(test_ids) != len(sample_ids) or set(test_ids["row_id"]) != set(sample_ids["row_id"]):
    raise ValueError("test와 sample_submission의 row_id가 일치하지 않습니다.")
display(train_check.groupby(["season", "game_type"])["control_success"].agg(["size", "mean"]))
print("test rows:", f"{len(test_ids):,}", "| sample rows:", f"{len(sample_ids):,}")

### 5. 전체 학습 및 `submit_02.zip` 생성

학습 전 학습용·평가용 피처가 정확히 같은지 확인한다. 학습 후에는 native 모델 저장 전후 예측, 행 순서 변경, 분할, 단일 행 추론, 실제 test 전체 실행, ZIP 구조와 손상 여부를 자동 검증한다.

In [ ]:
cmd = [
    sys.executable, str(PROJECT_ROOT / "experiments/05c_train_submit.py"),
    "--train-path", str(PROJECT_ROOT / "data/train.csv"),
    "--data-dir", str(PROJECT_ROOT / "data"),
    "--output-dir", str(OUTPUT_DIR),
    "--oof-latest-path", str(OOF_LATEST_PATH),
    "--threads", str(THREADS),
    "--lgbm-estimators", str(LGBM_ESTIMATORS),
    "--catboost-iterations", str(CATBOOST_ITERATIONS),
    "--catboost-task-type", CATBOOST_TASK_TYPE,
]
subprocess.run(cmd, check=True)

### 6. 학습 결과와 자동 검증 확인

In [ ]:
SUMMARY_PATH = OUTPUT_DIR / "training_summary.json"
if not SUMMARY_PATH.exists():
    raise FileNotFoundError("training_summary.json이 없습니다. 5번 셀을 완료하세요.")
with SUMMARY_PATH.open(encoding="utf-8") as handle:
    summary = json.load(handle)

model_rows = []
for name in ["lgbm", "catboost"]:
    row = {"model": name, **summary[name]}
    row.pop("parameters", None)
    model_rows.append(row)
display(pd.DataFrame(model_rows)[[
    "model", "feature_bundle", "train_rows", "train_game_type",
    "target_rate", "half_life", "feature_count",
    "categorical_count", "train_seconds",
]])
print("feature parity:", summary["training_inference_feature_parity"])
print("serialization/row audit:", summary["audit"])
print("end-to-end:", summary["end_to_end"])
if summary["training_inference_feature_parity"]["status"] != "passed":
    raise RuntimeError("학습/평가 피처 일치 검사를 통과하지 못했습니다.")

### 7. 최종 피처 중요도 확인

전체 데이터 재학습 후 중요도는 OOF 선택을 다시 바꾸는 용도가 아니라, 선택한 새 피처가 실제 최종 모델에서 사용됐는지 확인하는 감사 자료다.

In [ ]:
lgbm_importance = pd.read_csv(OUTPUT_DIR / "lgbm_feature_importance.csv")
catboost_importance = pd.read_csv(OUTPUT_DIR / "catboost_feature_importance.csv")
print("LightGBM importance top 25")
display(lgbm_importance.head(25))
print("CatBoost importance top 25")
display(catboost_importance.head(25))

### 8. 제출 ZIP 구조·파일명·해시 확인

ZIP 최상위에는 `model/`, `script.py`, `requirements.txt`만 있어야 한다. `submit_02.zip`은 확장자를 포함해 13자로 DACON 30자 제한을 충족한다.

In [ ]:
SUBMIT_ZIP = Path(summary["package"]["zip_path"])
EXPECTED = {
    "model/lgbm_model.txt",
    "model/catboost_model.cbm",
    "model/artifacts.joblib",
    "script.py",
    "requirements.txt",
}
with zipfile.ZipFile(SUBMIT_ZIP) as archive:
    names = set(archive.namelist())
    corrupt = archive.testzip()
    for info in archive.infolist():
        print(f"{info.filename:30s} {info.file_size / 1e6:9.3f} MB")
if names != EXPECTED:
    raise RuntimeError(f"ZIP 구조 오류: missing={EXPECTED-names}, extra={names-EXPECTED}")
if corrupt is not None:
    raise RuntimeError(f"손상된 ZIP 멤버: {corrupt}")
if len(SUBMIT_ZIP.name) > 30:
    raise RuntimeError("제출 파일명이 30자를 초과합니다.")
print("\nSubmission ZIP:", SUBMIT_ZIP)
print("filename length:", len(SUBMIT_ZIP.name))
print("ZIP size (MB):", round(SUBMIT_ZIP.stat().st_size / 1e6, 3))
print("SHA-256:", summary["package"]["zip_sha256"])

### 9. 두 번째 제출과 해석

DACON에는 `submit_02.zip` 자체를 압축 해제하지 않고 업로드한다. 첫 제출 Public은 856.9041이며, 새 후보의 최근 OOF BSS는 597.8865에서 609.2417로 개선됐다. 두 점수의 절대값 분포가 다르므로 리더보드가 정확히 11.36점 오른다고 가정하지 않는다. 이번 제출은 OOF 방향이 2025 test에서도 재현되는지 확인하는 체크포인트다.

In [ ]:
comparison = pd.DataFrame([
    {
        "version": "submit_01",
        "lgbm_features": "base",
        "catboost_features": "base",
        "recent_oof_bss": 597.886494,
        "public_score": 856.904064,
    },
    {
        "version": "submit_02",
        "lgbm_features": "pitchmix_shape",
        "catboost_features": "all_new",
        "recent_oof_bss": summary["validation_reference"]["bss_raw"],
        "public_score": np.nan,
    },
])
display(comparison)
print("업로드할 파일:", SUBMIT_ZIP)

### 10. 검토용 결과 ZIP

아래 ZIP은 DACON 제출용이 아니다. 제출 결과 검토에 필요한 학습 요약과 피처 중요도만 담는다.

In [ ]:
review_files = [
    OUTPUT_DIR / "training_summary.json",
    OUTPUT_DIR / "lgbm_feature_importance.csv",
    OUTPUT_DIR / "catboost_feature_importance.csv",
]
REVIEW_ZIP = OUTPUT_DIR / "submit02_review.zip"
with zipfile.ZipFile(REVIEW_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in review_files:
        if not path.exists():
            raise FileNotFoundError(path)
        archive.write(path, arcname=path.name)
print("review ZIP:", REVIEW_ZIP)
print("review size (MB):", round(REVIEW_ZIP.stat().st_size / 1e6, 3))